In [1]:
# SETUP — do not change this cell. Run it once before you start (and again any time you want fresh data).
with open('products.csv', 'w') as f:
    f.write('P01,Notebook A5,25,40\n'
            'P02,Blue Pen,10,100\n'
            'P03,Highlighter,18,12\n'
            'P04,USB Flash 32GB,150,6\n'
            'P05,Calculator,320,1\n'
            'P06,Sticky Notes,22,30\n'
            'P07,Python Book,450,3\n')

with open('sales.txt', 'w') as f:
    f.write('P02,12\nP01,3\nP05,1\n P04 , 2\nP09,1\nP07,5\nP02,8\n'
            'P03,4\nP01,10\nP06,2\np03,1\nP04,3\nP07,2\nP02,abc\n')

with open('daily_log.txt', 'w') as f:
    f.write('2026-10-15,Somchai,1840.00\n')

print('Files ready: products.csv, sales.txt, daily_log.txt')

Files ready: products.csv, sales.txt, daily_log.txt


In [3]:
# ============================================================
#  Week 9 Assignment — Siam Campus Shop: End-of-Day System
#  Name: ____________________   Student ID: 6805140024
#  Write ALL of your code in THIS cell only.
# ============================================================


def load_products(filename):
    """Read products.csv and return a dictionary:
       code -> [name, price, stock]      e.g. 'P01' -> ['Notebook A5', 25, 40]
       price and stock must be int."""
    products = {}
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            code = parts[0].strip()
            name = parts[1].strip()
            price = int(parts[2])
            stock = int(parts[3])
            products[code] = [name, price, stock]
    return products


def line_price(price, qty):
    """Return the money for one sale line.
       If qty is 10 or more, give a 10% discount. Return a float."""
    total = float(price * qty)
    if qty >= 10:
        total = total * 0.9
    return total


def process_sales(filename, products):
    """Read sales.txt line by line and return three things:  sold, revenue, errors
       sold    : dictionary  code -> [total quantity, total money]   e.g. 'P02' -> [20, 188.0]
       revenue : total money (float)
       errors  : list of error messages (strings)
       Also reduce the stock in `products` for every valid sale."""
    sold = {}
    revenue = 0.0
    errors = []

    with open(filename) as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            parts = line.split(',')
            if len(parts) < 2:
                errors.append(f"Line {line_number}: invalid line '{line}'")
                continue

            code = parts[0].strip().upper()
            qty_text = parts[1].strip()

            try:
                quantity = int(qty_text)
            except ValueError:
                errors.append(f"Line {line_number}: invalid quantity '{qty_text}'")
                continue

            if code not in products:
                errors.append(f"Line {line_number}: unknown product code {code}")
                continue

            if quantity > products[code][2]:
                errors.append(
                    f"Line {line_number}: not enough stock for {code} "
                    f"(want {quantity}, have {products[code][2]})"
                )
                continue

            total_money = line_price(products[code][1], quantity)

            if code not in sold:
                sold[code] = [quantity, total_money]
            else:
                sold[code][0] += quantity
                sold[code][1] += total_money

            revenue += total_money
            products[code][2] -= quantity

    return sold, revenue, errors


def best_seller(sold):
    """Return the code of the product with the highest total quantity sold."""
    top_product = ""
    highest_sold = 0
    for code in sold:
        if sold[code][0] > highest_sold:
            highest_sold = sold[code][0]
            top_product = code
    return top_product


def stock_status(stock):
    """Return 'OUT' if stock is 0, 'LOW' if stock is less than 5, otherwise 'OK'."""
    if stock == 0:
        return "OUT"
    elif stock < 5:
        return "LOW"
    else:
        return "OK"


def save_stock(filename, products):
    """Write the updated products to filename, in the same format as products.csv."""
    with open(filename, 'w') as f:
        for code in products:
            name, price, stock = products[code]
            f.write(f"{code},{name},{price},{stock}\n")


# ---------------- main program ----------------
cashier = input("Enter name: ")
date = input("Date: ")
products = load_products("products.csv")
sold, revenue, errors = process_sales("sales.txt", products)

print("=" * 48)
print(" SIAM CAMPUS SHOP - END OF DAY REPORT")
print(f" Date: {date}    Cashier: {cashier}")
print("=" * 48)
print(f"{'Code':<6}{'Product':<22}{'Sold':>8}{'Money':>12}")
for code in sorted(sold):
    name = products[code][0]
    qty = sold[code][0]
    money = sold[code][1]
    print(f"{code:<6}{name:<22}{qty:>8}{money:>12.2f}")

print("-" * 48)
print(f"Total revenue: {revenue:.2f} baht")

if sold:
    best_code = best_seller(sold)
    best_name = products[best_code][0]
    best_qty = sold[best_code][0]
    print(f"Best seller: {best_name} ({best_qty} sold)")
else:
    print("Best seller: none")

print()
print(f"Problems Found: {len(errors)}")
for error in errors:
    print(f"      {error}")

print()
print("-" * 48)
print("STOCK STATUS")
print(f"{'Code':<6}{'Product':<22}{'Stock':>8}{'Status':>12}")

for code in sorted(products):
    name = products[code][0]
    stock = products[code][2]
    status = stock_status(stock)
    print(f"{code:<6}{name:<22}{stock:>8}{status:>12}")

print()
print("-" * 48)
print("REORDER LIST (stock below 5)")
for code in sorted(products):
    if products[code][2] < 5:
        print(f"{code:<6}{products[code][0]:<22}{products[code][2]:>8}")

print()
save_stock("products_updated.csv", products)
print("Updated stock saved to products_updated.csv")

with open("daily_log.txt", "a") as f:
    f.write(f"{date},{cashier},{revenue:.2f}\n")

print("Daily log updated.")

Enter name: Paleon
Date: 21/10/26
 SIAM CAMPUS SHOP - END OF DAY REPORT
 Date: 21/10/26    Cashier: Paleon
Code  Product                   Sold       Money
P01   Notebook A5                 13      300.00
P02   Blue Pen                    20      188.00
P03   Highlighter                  5       90.00
P04   USB Flash 32GB               5      750.00
P05   Calculator                   1      320.00
P06   Sticky Notes                 2       44.00
P07   Python Book                  2      900.00
------------------------------------------------
Total revenue: 2592.00 baht
Best seller: Blue Pen (20 sold)

Problems Found: 3
      Line 5: unknown product code P09
      Line 6: not enough stock for P07 (want 5, have 3)
      Line 14: invalid quantity 'abc'

------------------------------------------------
STOCK STATUS
Code  Product                  Stock      Status
P01   Notebook A5                 27          OK
P02   Blue Pen                    80          OK
P03   Highlighter             